# Chapter 3: Prompt Engineering with MLflow

## 📓 About this notebook
This notebook manages Unity Airways customer-support prompts as governed assets in the MLflow Prompt Registry: creating and versioning prompts, promoting them with aliases, using them in code, and evaluating and optimizing them.

**Maps to the book:** Chapter 3, *Prompt Engineering with MLflow* — sections: Fundamentals of Prompt Engineering, Managing Prompts with the MLflow Prompt Registry, Using Prompts in Code, Evaluating and Comparing Prompt Versions, Prompt Optimization, Integrating Prompts into Applications.

### ✅ Prerequisites

- A Databricks workspace with **serverless compute** and a **pay-per-token Foundation Model** endpoint (examples use `databricks-gpt-oss-120b`).
- Unity Catalog permissions to create prompts in the catalog/schema set in [`conf/data.yml`](../conf/data.yml).

See the [repository README](../README.md) for full setup.

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

### Managing prompts in the Prompt Registry
Register prompts as named Unity Catalog entities with immutable versions, then promote them safely using mutable aliases such as staging and production. _(see Ch 3, "Managing Prompts with the MLflow Prompt Registry")_

In [0]:
import mlflow
mlflow.__version__

## Naming prompts and choosing a location

In [ ]:
# Set the Unity Catalog and schema for Unity Airways data and models
# These variables are used to reference tables and models throughout the notebook
import yaml
with open('../conf/data.yml') as f:
    default_uc = yaml.safe_load(f)['default_uc']
CATALOG = default_uc['catalog']
SCHEMA = default_uc['schema']

In [0]:
import mlflow

# Configure MLflow tracking and choose an experiment for your project
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/unity-airways/unity-airways-prompts")

# Link this experiment to a default UC schema for prompts
mlflow.set_experiment_tags({
    "mlflow.promptRegistryLocation": f"{CATALOG}.{SCHEMA}"
})

## Creating prompts programmatically with the Python SDK

A Unity Airways-flavored example:

In [0]:
import mlflow

uc_prompt = f"{CATALOG}.{SCHEMA}.unity_airways_customer_support"

v1 = mlflow.genai.register_prompt(
    name=uc_prompt,
    template="""\
You are a customer support assistant for Unity Airways.

Rules:
- If key details are missing, ask exactly one clarifying question.
- Do not invent fees, waivers, or exceptions.

Customer question: {{question}}

Write a concise answer (max 120 words).
""",
    commit_message="v1: baseline support answer with safety and brevity constraints",
    tags={
        "use_case": "customer_support",
        "language": "en",
        "owner": "unity-airways-support",
    },
)

print(f"Created prompt {v1.name} version {v1.version}")

## Versioning prompts

Here is what a “small, intentional change” looks like:

In [0]:
import mlflow

uc_prompt = f"{CATALOG}.{SCHEMA}.unity_airways_customer_support"

v2 = mlflow.genai.register_prompt(
    name=uc_prompt,
    template="""\
You are a customer support assistant for Unity Airways.

Rules:
- If the question is ambiguous, ask exactly one clarifying question.
- If the customer mentions refunds, do not promise eligibility without fare details.
- Do not invent fees, waivers, or exceptions.
- Keep the answer under 120 words.

Customer question:
{{question}}

Answer:
""",
    commit_message="v2: tighten ambiguity handling and refund safety posture",
    tags={
        "change_type": "behavior",
        "risk": "medium",
        "hypothesis": "reduces overconfident refund promises",
    },
)

print(f"Created version {v2.version} of {v2.name}")

## Promoting via aliases

Setting an alias is a simple operation:

In [0]:
import mlflow
mlflow.genai.set_prompt_alias(
    name=f"{CATALOG}.{SCHEMA}.unity_airways_customer_support",
    alias="staging",
    version=2
)
mlflow.genai.set_prompt_alias(
    name=f"{CATALOG}.{SCHEMA}.unity_airways_customer_support",
    alias="production",
    version=1
)

## Searching prompts in Unity Catalog

Here is the pattern Databricks demonstrates:

In [0]:
import mlflow

# Required format: list all prompts in a catalog.schema
all_prompts = mlflow.genai.search_prompts(f"catalog = '{CATALOG}' AND schema = '{SCHEMA}'")

# Filter programmatically
ua_prompts = [p for p in all_prompts if "unity_airways" in p.name.lower()]
support_prompts = [p for p in all_prompts if p.tags.get("use_case") == "customer_support"]

In [0]:
support_prompts

## Deleting prompts

That means cleanup typically looks like this:

In [0]:
from mlflow import MlflowClient
client = MlflowClient()
prompt_name = f"{CATALOG}.{SCHEMA}.unity_airways_customer_support"

# Delete specific versions first (required for Unity Catalog)
#client.delete_prompt_version(prompt_name, "1")
#client.delete_prompt_version(prompt_name, "2")

# Then delete the prompt itself
#client.delete_prompt(prompt_name)

## Loading a prompt by version

The URI form is easy to standardize:

## Using prompts in code

Load prompts from the registry by version (for reproducibility) or by alias (for production), and render their template variables safely before calling the model. _(see Ch 3, "Using Prompts in Code")_

In [0]:
import mlflow

prompt_name = f"{CATALOG}.{SCHEMA}.unity_airways_customer_support"
prompt_v2 = mlflow.genai.load_prompt(f"prompts:/{prompt_name}/2")

print(prompt_v2.name, prompt_v2.version)

## Loading a prompt by alias

The production path is defined by aliases. Databricks specifies the alias URI syntax as prompts:/{catalog}.{schema}.{prompt_name}@{alias}.

In [0]:
import mlflow

prompt_name = f"{CATALOG}.{SCHEMA}.unity_airways_customer_support"
prompt_prod = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@production")

print(prompt_prod.name, prompt_prod.version)

## Loading a prompt by alias in applications

Here is that pattern, adapted to Unity Airways:

In [0]:
import os
import mlflow

mlflow.set_registry_uri("databricks-uc")

def load_runtime_prompt() -> object:
    prompt_alias = os.getenv("PROMPT_ALIAS", "production")
    prompt_uri = os.getenv("PROMPT_URI", f"{CATALOG}.{SCHEMA}.unity_airways_customer_support")
    uri = f"prompts:/{prompt_uri}@{prompt_alias}"
    return mlflow.genai.load_prompt(uri)

prompt = load_runtime_prompt()
print(f"Loaded {prompt.name} v{prompt.version}")

## Rendering template variables with format()

A small wrapper makes this safer:

In [0]:
from typing import Any, Dict

def render_prompt(prompt_obj, variables: Dict[str, Any]) -> str:
    try:
        return prompt_obj.format(**variables)
    except Exception as e:
        raise ValueError(
            f"Prompt formatting failed for '{getattr(prompt_obj, 'name', 'unity_airways_customer_support')}' "
            f"(version={getattr(prompt_obj, 'version', '1')}). "
            f"Provided keys: {sorted(list(variables.keys()))}"
        ) from e

text = render_prompt(prompt, {"question": "Do I get a refund if I miss my flight?"})

In [0]:
print(text)

## Making the model call: a minimal Databricks pattern

Here is a minimal end-to-end call:

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

prompt = mlflow.genai.load_prompt(f"prompts:/{CATALOG}.{SCHEMA}.unity_airways_customer_support@production")
content = prompt.format(question="Can I change my flight tomorrow?")
resp = client.chat.completions.create(
    model="databricks-gpt-oss-120b",
    messages=[{"role": "user", "content": content}],
    temperature=0.1,
    max_tokens=350,
)
answer = resp.choices[0].message.content
print(answer)

## Building a prompt evaluation dataset

Here is a dataset creation pattern that stores the dataset in Unity Catalog and populates it with a few initial cases:

## Evaluating and comparing prompt versions

Build an evaluation dataset of expected facts, then score prompt versions side-by-side with MLflow's evaluation framework to decide objectively which one to promote. _(see Ch 3, "Evaluating and Comparing Prompt Versions")_

In [0]:
EVAL_DATASET_NAME = f"{CATALOG}.{SCHEMA}.ua_support_prompt_eval"
eval_dataset = mlflow.genai.datasets.create_dataset(
    name=EVAL_DATASET_NAME
)
evaluation_examples = [
    {
        "inputs": {
            "question": "My flight is tomorrow. Can I change it to next week?"
        },
        "expectations": {
            "expected_facts": [
                "Eligibility depends on fare rules or fare type",
                "May involve a change fee or fare difference",
                "Ask for booking reference or fare details if missing"
            ]
        }
    },
    {
        "inputs": {
            "question": "I missed my flight due to traffic. Do I get a refund?"
        },
        "expectations": {
            "expected_facts": [
                "Refund eligibility depends on fare rules",
                "Do not promise a refund without checking ticket conditions",
                "Provide next steps to verify eligibility"
            ]
        }
    },
    {
        "inputs": {
            "question": "My flight was canceled. Can I rebook for free?"
        },
        "expectations": {
            "expected_facts": [
                "Rebooking depends on disruption policy",
                "Ask for booking details if needed",
                "Avoid claiming blanket waivers without evidence"
            ]
        }
    },
]
eval_dataset = eval_dataset.merge_records(evaluation_examples)
print(f"Added {len(evaluation_examples)} examples to {EVAL_DATASET_NAME}")

## Creating prompt versions to compare

Here is a simple example of registering two versions under the same prompt name:

In [0]:
prompt_name

## Creating a predict function

Now we need a consistent way to run each prompt version. The pattern is always the same: load a specific prompt version from Prompt Registry, render variables with .format(...), send rendered text to an LLM, and return the output in a consistent shape so evaluation can score it.

We will use Databricks model serving exclusively via the OpenAI-compatible client provided by the Databricks SDK.

In [0]:
client = w.serving_endpoints.get_open_ai_client()

def create_support_function(prompt_name: str, version: int):
    """
    Returns a predict_fn for a specific prompt version.
    The function returns a dict so evaluation can read outputs consistently.
    """
    def answer_question(question: str) -> dict:
        prompt = mlflow.genai.load_prompt(
            name_or_uri=f"prompts:/{prompt_name}/{version}"
        )
        formatted = prompt.format(question=question)

        resp = client.chat.completions.create(
            model="databricks-gpt-oss-120b",
            messages=[{"role": "user", "content": formatted}],
            temperature=0.1,
            max_tokens=350,
        )
        return {"response": resp.choices[0].message.content}

    return answer_question

## Running comparative evaluation and scoring

Next, run evaluation for each version. The "checks" you choose should be minimal at first. A good starting point is correctness against expected facts because it aligns directly with your dataset structure.

In [0]:
from mlflow.genai.scorers import Correctness

checks = [
    Correctness(),  # uses expected_facts in the dataset expectations
]

results = {}
for version in [v1.version, v2.version]:
    print(f"Evaluating version {version}")
    with mlflow.start_run(run_name=f"ua_support_v{version}_eval"):
        mlflow.log_param("prompt_name", prompt_name)
        mlflow.log_param("prompt_version", version)
        mlflow.log_param("eval_dataset", EVAL_DATASET_NAME)

        eval_results = mlflow.genai.evaluate(
            predict_fn=create_support_function(prompt_name, version),
            data=eval_dataset,
            scorers=checks,
        )

        results[f"v{version}"] = eval_results
        print(f"Correctness: {eval_results.metrics.get('correctness/mean', 0):.2f}")

## Compare programmatically

If you want a quick numeric summary in notebooks or CI-style checks, keep the programmatic comparison. Start simple with a single metric, then add more checks only when you have a clear reason.

In [0]:
print("\n=== Version Comparison ===")
for version_label, result in results.items():
    correctness = result.metrics.get("correctness/mean", 0)
    print(f"{version_label}: correctness={correctness:.2f}")

best = max(results.items(), key=lambda kv: kv[1].metrics.get("correctness/mean", 0))
print(f"\nBest version by correctness: {best[0]}")

## Step 1: Register a baseline prompt

Establish a simple, baseline prompt version to serve as the "current behavior" and stable reference point. This initial prompt needs minimum guardrails (e.g., do not speculate on fees, ask for missing details, keep it short). Once registered, the baseline is versioned and immutable, allowing meaningful comparison for future changes. Each improvement becomes a new, clearly committed version.

## Prompt Optimization

Prompt optimization is an automated way to improve a prompt template using data and measurable criteria. Instead of guessing which wording will help, you provide (1) representative inputs, (2) what a good answer must contain, and (3) a budget for exploration. The optimizer proposes an improved prompt template. You still review it, register it as a new version, and promote it only after it performs well in your version comparison workflow.

_(see Ch 3, "Prompt Optimization")_

In [0]:
PROMPT_NAME = f"{CATALOG}.{SCHEMA}.unity_airways_support_answer"

baseline = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template="""\
You are a customer support assistant for Unity Airways.

Rules:
- If key details are missing, ask exactly one clarifying question.
- Do not invent fees, waivers, or refund eligibility.
- Keep the answer under 120 words.

Customer question:
{{question}}

Answer:
""",
    commit_message="v1: baseline support answer prompt"
)

print(f"Baseline prompt: {baseline.name} v{baseline.version}")

## Step 2: Define a prediction function

This function loads a specific prompt version, formats it, and sends it to a model endpoint via the OpenAI-compatible client from the Databricks SDK.

In [0]:
from databricks_openai import DatabricksOpenAI

openai_client = DatabricksOpenAI()

# Define your prediction function
def predict_fn(question: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{baseline.version}")
    content = prompt.format(question=question)

    resp = openai_client.chat.completions.create(
        model="databricks-gpt-oss-120b",
        messages=[{"role": "user", "content": content}],
        temperature=0.1,
        max_tokens=350,
    )
    return resp.choices[0].message.content

In [ ]:
from IPython.display import Markdown

output = predict_fn("My flight is tomorrow. Can I change it to next week?")

Markdown(output)

## Step 3: Provide training examples with expected facts

Training examples should look like the questions you actually get, not like the questions you wish customers would ask. Include edge cases where the model is tempted to guess.

In [0]:
train_dataset = [
    {
        "inputs": {"question": "My flight is tomorrow. Can I change it to next week?"},
        "expectations": {
            "expected_facts": [
                "Eligibility depends on fare rules or fare type",
                "May involve a change fee or fare difference",
                "Ask for booking reference or fare details if missing"
            ]
        }
    },
    {
        "inputs": {"question": "I missed my flight due to traffic. Do I get a refund?"},
        "expectations": {
            "expected_facts": [
                "Refund eligibility depends on fare rules",
                "Do not promise a refund without checking ticket conditions",
                "Provide next steps to verify eligibility"
            ]
        }
    },
    {
        "inputs": {"question": "My flight was canceled. Can I rebook for free?"},
        "expectations": {
            "expected_facts": [
                "Rebooking depends on disruption policy",
                "Ask for booking details if needed",
                "Avoid claiming blanket waivers without evidence"
            ]
        }
    },
    {
        "inputs": {"question": "Can I upgrade my seat after booking?"},
        "expectations": {
            "expected_facts": [
                "Seat upgrades depend on availability and fare rules",
                "May require additional payment",
                "Ask for booking reference or fare type"
            ]
        }
    },
    {
        "inputs": {"question": "What documents do I need for international travel?"},
        "expectations": {
            "expected_facts": [
                "Passport and visa requirements depend on destination",
                "Check government travel advisories",
                "Ask for destination details if missing"
            ]
        }
    },
    {
        "inputs": {"question": "How do I add extra baggage to my booking?"},
        "expectations": {
            "expected_facts": [
                "Extra baggage can be added online or at the airport",
                "Fees depend on route and fare type",
                "Ask for booking reference or baggage details"
            ]
        }
    },
    {
        "inputs": {"question": "Is there a fee for changing my name on the ticket?"},
        "expectations": {
            "expected_facts": [
                "Name changes may incur a fee",
                "Eligibility depends on fare rules",
                "Ask for booking reference and reason for change"
            ]
        }
    },
    {
        "inputs": {"question": "Can I request a special meal for my flight?"},
        "expectations": {
            "expected_facts": [
                "Special meals can be requested in advance",
                "Availability depends on route and airline policy",
                "Ask for booking reference and meal preference"
            ]
        }
    },
    {
        "inputs": {"question": "What happens if my flight is delayed overnight?"},
        "expectations": {
            "expected_facts": [
                "Overnight delays may entitle you to accommodation",
                "Check airline disruption policy",
                "Ask for booking details and delay duration"
            ]
        }
    },
    {
        "inputs": {"question": "How do I check in online for my flight?"},
        "expectations": {
            "expected_facts": [
                "Online check-in is available 24-48 hours before departure",
                "Requires booking reference and passenger details",
                "Provide steps or link to check-in portal"
            ]
        }
    }
]

## Step 4: Run optimization and inspect the candidate template

Run optimization and extract the candidate template:

In [ ]:
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=train_dataset,
    prompt_uris=[baseline.uri],
    optimizer=GepaPromptOptimizer(reflection_model="databricks:/databricks-gpt-oss-120b", max_metric_calls=20),
    scorers=[Correctness(model="databricks:/databricks-gpt-oss-120b")],
)

candidate = result.optimized_prompts[0]
print("=== Candidate optimized template ===")
print(candidate.template)

## Publishing versions and assigning aliases

The following snippet creates two versions of a Unity Airways support prompt and assigns aliases so staging points to the newer candidate while production remains on the known-good version:

## Integrating prompts into applications

Publish versions and point aliases at them so applications load prompts by alias, letting you promote or roll back behavior without redeploying code. _(see Ch 3, "Integrating Prompts into Applications")_

In [0]:
PROMPT = f"{CATALOG}.{SCHEMA}.unity_airways_customer_support"

# Version 1: baseline
v1 = mlflow.genai.register_prompt(
    name=PROMPT,
    template="""\
You are a Unity Airways customer support assistant.
Customer question: {{question}}
""",
    commit_message="v1: minimal support prompt",
)

# Version 2: safer, more constrained behavior
v2 = mlflow.genai.register_prompt(
    name=PROMPT,
    template="""\
You are a careful Unity Airways customer support assistant.

Rules:
- If key details are missing, ask exactly one clarifying question.
- Do not invent fees, waivers, or refund eligibility.
- Keep the answer under 120 words.

Customer question:
{{question}}

Answer:
""",
    commit_message="v2: add safety constraints and brevity limit",
)

# Aliases control rollout
mlflow.genai.set_prompt_alias(name=PROMPT, alias="staging", version=v2.version)
mlflow.genai.set_prompt_alias(name=PROMPT, alias="production", version=v1.version)

print(f"Staging -> v{v2.version}, Production -> v{v1.version}")

## End-to-end “thin slice”

The example below shows an end-to-end function that answers a customer question using whatever alias is configured in the environment.

In [ ]:
import os
from databricks_openai import DatabricksOpenAI

openai_client = DatabricksOpenAI()

def answer_customer(question: str) -> str:
    prompt_name = os.getenv("PROMPT_URI", f"{CATALOG}.{SCHEMA}.unity_airways_customer_support")
    alias = os.getenv("PROMPT_ALIAS", "production")

    # Load the prompt by alias
    prompt = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@{alias}")

    # Render the template variables
    content = prompt.format(question=question)

    # Call the model serving endpoint
    resp = openai_client.chat.completions.create(
        model="databricks-gpt-oss-120b",
        messages=[{"role": "user", "content": content}],
        temperature=0.1,
        max_tokens=350,
    )
    return resp.choices[0].message.content

print(answer_customer("My flight is tomorrow. Can I change it to next week?"))

## Promoting to production and rolling back safely

First, promote by moving the production alias to the candidate version:

In [0]:
import mlflow

mlflow.genai.set_prompt_alias(
    name=f"{CATALOG}.{SCHEMA}.unity_airways_customer_support",
    alias="production",
    version=v2.version
)

## Rolling back safely

If anything unexpected happens, rollback is equally simple. Move production back to the previous version:

In [0]:
mlflow.genai.set_prompt_alias(
    name=f"{CATALOG}.{SCHEMA}.unity_airways_customer_support",
    alias="production",
    version=v1.version
)